In [35]:
from openai import OpenAI

client = OpenAI()

MODEL = "gpt-6-luna"

customers = {
    "a123": {
        "name": "Alex",
        "plan": "Premium",
    },
    "b456": {
        "name": "Anna",
        "plan": "Free",
    },
}


def get_customer(customer_id):
    return customers[customer_id]


tools = {
    "get_customer": get_customer,
}


agent = client.beta.agents.create(
    model=MODEL,
    instructions="You speak short sentences. Like caveman.",
    reasoning={"effort": "low"},
    text={"verbosity": "low"},
    tools=[
        {
            "type": "web_search",
            "mode": "live",
            "location": {
                "country": "UAE",
            },
        },
        {
            "type": "function",
            "name": "get_customer",
            "description": "Look up a customer by ID.",
            "parameters": {
                "type": "object",
                "properties": {"customer_id": {"type": "string"}},
                "required": ["customer_id"],
                "additionalProperties": False,
            },
        },
    ],
)

In [36]:
with client.beta.agents.sessions.create(
    agent_id=agent.id,
    environment={"type": "none"},
    input="What is the plan of customer a123?",
    stream=True,
) as events:
    for event in events:
        if event.type == "agent.session.created":
            session_id = event.session.id
        elif event.type == "agent.session.requires_action":
            print(event.session.required_actions)
            break
        elif event.type == "agent.session.turn.output_text.done":
            print(event.text)
        elif (
            event.type == "agent.session.turn.item.done"
            and event.item.type == "web_search_call"
        ):
            print("Search: ", event.item.action)
        else:
            print(event.type)

agent.session.turn.created
agent.session.turn.item.added
agent.session.in_progress
agent.session.turn.in_progress
agent.session.turn.item.added
[RequiredActionSessionRequiredActionResourceFunctionCall(arguments={'customer_id': 'a123'}, call_id='exec_a918535c2a6b6bef94fd259ef179156afaef57fad1134f35dc', name='get_customer', turn_id='turn_0a2b959b1d97d49e006ae2e7c8591c819a9f99edc08015631c', type='function_call')]


In [37]:
session = client.beta.agents.sessions.retrieve(session_id)

for action in session.required_actions:
    print(action.to_dict())

session.status

{'arguments': {'customer_id': 'a123'}, 'call_id': 'exec_a918535c2a6b6bef94fd259ef179156afaef57fad1134f35dc', 'name': 'get_customer', 'turn_id': 'turn_0a2b959b1d97d49e006ae2e7c8591c819a9f99edc08015631c', 'type': 'function_call'}


'requires_action'

In [ ]:
import json

with client.beta.agents.sessions.events.stream(session_id) as events:
    for action in session.required_actions:
        action = action.to_dict()
        output = tools[action["name"]](action["arguments"]["customer_id"])
        client.beta.agents.sessions.events.create(
            session_id=session_id,
            events=[
                {
                    "type": "agent.session.input.tool_result",
                    "output": json.dumps(output),
                    "success": True,
                    "call_id": action["call_id"],
                    "turn_id": action["turn_id"],
                }
            ],
        )

        for event in events:
            if event.type == "agent.session.turn.output_text.done":
                print(event.text)
            else:
                print(event.type)

agent.session.turn.item.added
agent.session.turn.item.added
agent.session.in_progress
agent.session.turn.item.done
agent.session.turn.item.added
agent.session.turn.item.added
agent.session.turn.content_part.added
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
Customer a123 (Alex) is on the Premium plan.
agent.session.turn.content_part.done
agent.session.turn.item.done
agent.session.turn.completed
agent.session.idle
